# sheet_operate — Phase 4.5 DPO（偏好對齊：不只要對，方法也要對）

**位置：SFT → DPO → GRPO**

為什麼 DPO 排在 GRPO 之前：這個專案已經觀察過兩次「SFT 洗掉 RL 成果」
（v2 93.8→75.0、sft_v4 95.8→58.3）。DPO 同樣會位移 policy，所以放在 RL 之前；
GRPO 擺最後，它的成果才不會被後續訓練沖掉。

**DPO 補的是 GRPO reward 補不到的東西。** GRPO 的 reward 是純輸出導向
（`score + 1.0 if full_match`），一段硬編 `row[7]` 剛好猜對照樣拿滿分——
SFT 階段辛苦過濾掉的壞習慣，會在 RL 階段被重新獎勵回來。

DPO 的資料正是為此準備的（`scripts/build_dpo.py`）：

| 負例來源 | 佔比 | 含硬編索引 |
|---|---|---|
| A：未通過 Gym（答案錯／程式碼掛掉） | 64% | 31% |
| **B：通過 Gym 但方法錯**（方法檢查／泛化檢查剔除的） | 36% | **85%** |

B 類是關鍵：它們逐格比對全對，卻示範了真實檔案上會靜默失敗的寫法。
只用 A 類，模型學到「不要出錯」；加上 B 類才學到「即使會對也不要這樣寫」。


In [ ]:
# ===== 設定 =====
CFG = dict(
    repo_url    = "https://github.com/timmytsaa/sheet_operate.git",
    drive_root  = "/content/drive/MyDrive/公司/未命名資料夾/sheet_operate",
    run_name    = "dpo_v1",
    sft_adapter = "adapter_sft_v5",      # 起點：SFT 完成的 adapter（資料夾或含 checkpoint-N）
    base_model  = "Qwen/Qwen3-4B-Instruct-2507",
    dpo_files   = ["data/sft/dpo_v6v7.jsonl"],
    max_seq_len = 6144,
    lora_r = 64, lora_alpha = 64,
    # --- DPO ---
    beta   = 0.1,        # 偏好強度。資料只有 ~300 對，調高容易過擬合
    lr     = 5e-6,       # 比 SFT 低一個量級
    epochs = 2,
    per_device_bs = 2, grad_accum = 8,
    max_prompt_len = 3584, max_len = 6144,
    eval_limit = None,
    seed = 3407,
)

In [ ]:
%%capture
# ===== 安裝依賴 =====
!pip install unsloth trl peft
!pip install openpyxl formulas   # formulas：公式任務的評測靠它求值，缺了會低估分數

In [ ]:
# ===== 掛載 Drive、取得 repo =====
import glob, json as _json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

os.makedirs(CFG["drive_root"], exist_ok=True)
REPO = "/content/sheet_operate"
MARKER = os.path.join("scripts", "gen_tasks.py")

def has_marker(p):
    return os.path.exists(os.path.join(p, MARKER))

if CFG["repo_url"] and not has_marker(REPO):
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", CFG["repo_url"], REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=False)
    except Exception as e:
        print("[提示] git clone 失敗，改試 Drive 的 zip：", e)
    if not has_marker(REPO):
        shutil.rmtree(REPO, ignore_errors=True)

if not has_marker(REPO):
    hits = sorted(glob.glob(os.path.join(CFG["drive_root"], "*.zip")))
    if hits:
        import zipfile
        shutil.rmtree(REPO, ignore_errors=True)
        with zipfile.ZipFile(hits[0]) as z:
            z.extractall(REPO)

assert has_marker(REPO), "找不到可用的 repo"
sys.path.insert(0, REPO)
os.chdir(REPO)

def run_script(args):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError("指令失敗：" + " ".join(args))

# 評測用任務集（seed 與訓練隔離）
V6 = "dup_header,misaligned_merge,two_tier_header,pair_group"
V7 = "diff_dirty_key,diff_nullkey,diff_dupkey,diff_carry_cols,diff_multicol"
for out, n, seed, fams in (("data/tasks/eval_v6", 4, 900016, V6),
                           ("data/tasks/eval_v7", 4, 900017, V7)):
    if not os.path.isdir(out):
        run_script(["scripts/gen_tasks.py", "--out", out, "--n", str(n),
                    "--seed", str(seed), "--families", fams])
print("repo 就緒：", REPO)

In [ ]:
# ===== 載入模型 ＋ 接上 SFT adapter 當起點 =====
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["base_model"],
    max_seq_length = CFG["max_seq_len"],
    load_in_4bit   = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = CFG["lora_r"], lora_alpha = CFG["lora_alpha"], lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = CFG["seed"],
)

import glob as _g
from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict

def _find_adapter():
    """起點權重：確切資料夾，或含 checkpoint-N 的資料夾（取步數最大者）。"""
    roots = [CFG["drive_root"]] + sorted(
        _g.glob(os.path.join(os.path.dirname(CFG["drive_root"].rstrip("/")), "*")))
    for root in roots:
        base = os.path.join(root, CFG["sft_adapter"])
        direct = os.path.join(base, "adapter_model.safetensors")
        if os.path.exists(direct):
            return direct
        cks = _g.glob(os.path.join(base, "checkpoint-*", "adapter_model.safetensors"))
        if cks:
            def step(p):
                tail = os.path.basename(os.path.dirname(p)).rsplit("-", 1)[1]
                return int(tail) if tail.isdigit() else -1
            return max(cks, key=step)
    return None

adapter_path = _find_adapter()
assert adapter_path, "找不到起點 adapter：" + CFG["sft_adapter"] + "（請先跑完 SFT）"
set_peft_model_state_dict(model, load_file(adapter_path))
print("DPO 起點：", adapter_path)
# 參考模型 = 同一個模型停用 adapter（PEFT 標準做法，省一份權重的記憶體）
# 所以下面 DPOTrainer 的 ref_model 傳 None 即可

In [ ]:
# ===== 載入偏好對 =====
import collections, random
from datasets import Dataset
from sheetops.prompts import SYSTEM_PROMPT as _SYS

pairs = []
for fp in CFG["dpo_files"]:
    with open(fp, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = _json.loads(line)
            # 各批資料是不同時期蒸餾的，system 提示統一對齊到當前版本
            prompt = [{"role": "system", "content": _SYS}] + [
                m for m in r["prompt"] if m["role"] != "system"]
            pairs.append({"prompt": prompt, "chosen": r["chosen"],
                          "rejected": r["rejected"],
                          "family": r.get("family", "?"),
                          "neg_kind": r.get("neg_kind", "?")})

print("偏好對", len(pairs), "筆")
print("  負例來源：", dict(collections.Counter(p["neg_kind"] for p in pairs)))
print("  各家族：", dict(collections.Counter(p["family"] for p in pairs)))

random.seed(CFG["seed"])
random.shuffle(pairs)
train_ds = Dataset.from_list([{k: p[k] for k in ("prompt", "chosen", "rejected")}
                              for p in pairs])
print("訓練樣本：", len(train_ds))

In [ ]:
# ===== DPO 訓練（自動續訓） =====
from trl import DPOTrainer, DPOConfig
from transformers.trainer_utils import get_last_checkpoint

CKPT_DIR = os.path.join(CFG["drive_root"], "ckpt_" + CFG["run_name"])
os.makedirs(CKPT_DIR, exist_ok=True)

trainer = DPOTrainer(
    model = model,
    ref_model = None,                 # PeftModel：停用 adapter 即為參考模型
    args = DPOConfig(
        output_dir = CKPT_DIR,
        beta = CFG["beta"],
        learning_rate = CFG["lr"],
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.1,
        num_train_epochs = CFG["epochs"],
        per_device_train_batch_size = CFG["per_device_bs"],
        gradient_accumulation_steps = CFG["grad_accum"],
        max_prompt_length = CFG["max_prompt_len"],
        max_length = CFG["max_len"],
        logging_steps = 5,
        save_steps = 40,
        save_total_limit = 3,
        bf16 = True,
        report_to = "none",
        seed = CFG["seed"],
    ),
    train_dataset = train_ds,
    processing_class = tokenizer,
)

last_ckpt = get_last_checkpoint(CKPT_DIR)
if last_ckpt:
    print("=" * 70)
    print("找到既有 checkpoint：" + os.path.basename(last_ckpt) + "，將『續訓這一輪』。")
    print("    要開新一輪請把 CFG['run_name'] 換成沒用過的名字。")
    print("=" * 70)
trainer.train(resume_from_checkpoint = last_ckpt)
# 觀察重點：rewards/accuracies 應升到 0.7 以上；rewards/margins 穩定為正。
# 若 margins 爆衝而 accuracies 停滯 = 在鑽長度或格式的漏洞，要降 beta 或減 epoch。

In [ ]:
# ===== 存 DPO adapter =====
ADAPTER_DIR = os.path.join(CFG["drive_root"], "adapter_" + CFG["run_name"])
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to:", ADAPTER_DIR)
print("下一步：把 colab_grpo.ipynb 的 CFG['sft_adapter'] 改成", "adapter_" + CFG["run_name"])

In [ ]:
# ===== 訓後評測：pass@1 ＋ 方法合規率 =====
# DPO 要改的是「方法」，所以除了 pass@1，還要量硬編欄位索引的比例——
# 那正是 Gym 分數看不出來的東西。
sys.path.insert(0, os.path.join(REPO, "scripts"))
from merge_teacher import LITERAL_INDEX

from sheetops.encoder import encode_workbook
from sheetops.env import solve_once
from sheetops.executor import extract_code
from sheetops.prompts import SYSTEM_PROMPT, build_user_prompt

FastLanguageModel.for_inference(model)

def run_gym_eval(tasks_dir, tag):
    task_dirs = sorted(p.parent for p in Path(tasks_dir).glob("*/task.json"))
    if CFG["eval_limit"]:
        task_dirs = task_dirs[:CFG["eval_limit"]]
    rows = []
    for i, td in enumerate(task_dirs):
        spec = _json.loads((td / "task.json").read_text(encoding="utf-8"))
        user = build_user_prompt(spec["instruction"],
                                 encode_workbook(td / "start.xlsx"),
                                 spec.get("context", ""))
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
        code_ = extract_code(reply)
        rep = solve_once(td, code_) if code_ else {"full_match": False, "score": 0.0}
        rows.append({"id": spec["id"], "family": spec["family"],
                     "pass": bool(rep["full_match"]), "score": rep["score"],
                     "hardcoded": bool(code_ and LITERAL_INDEX.search(code_))})
        if (i + 1) % 12 == 0:
            print("  [" + tag + "] " + str(i + 1) + "/" + str(len(task_dirs))
                  + "  pass: " + str(sum(r["pass"] for r in rows)))
    import pandas as pd
    df = pd.DataFrame(rows)
    print("[" + tag + "] pass@1 = " + format(df["pass"].mean(), ".1%")
          + "｜硬編欄位索引 = " + format(df["hardcoded"].mean(), ".1%") + "（越低越好）")
    print(df.groupby("family")[["pass", "hardcoded"]].mean().to_string())
    return df

for d, tag in (("data/tasks/eval_v6", "v6 欄位定位"),
               ("data/tasks/eval_v7", "v7 差異比對"),
               ("data/tasks/eval",    "v1 基礎")):
    if os.path.isdir(d):
        run_gym_eval(d, tag)

## 之後

1. 把 `colab_grpo.ipynb` 的 `CFG["sft_adapter"]` 改成本輪的 `adapter_dpo_v1`，
   `CFG["run_name"]` 換成沒用過的名字（**同名會續訓舊 checkpoint，等於丟掉這輪的起點**）。
2. GRPO 跑完再匯出 GGUF（`colab_export_gguf.ipynb`）。

**判讀重點**：DPO 的目標不是把 pass@1 拉高，是把「硬編欄位索引」的比例壓下去。
如果 pass@1 持平但硬編率明顯下降，這輪就成功了——那代表模型改用查表頭名稱的寫法，
在真實檔案（欄名重複、表頭跨兩列）上才不會靜默失敗。
